# Mini-TP 7 · Seguridad con SAIF — Data Lake (MinIO/S3)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

Este es **uno de los seis notebooks guiados** del Mini-TP 7. Cada uno asegura **una capa**
de las que construimos en el bimestre. **Elige este si tu modelo se expone por Data Lake (MinIO/S3)**
(o simplemente para practicar esta estrategia). Con completar **uno** alcanza para la entrega.

### SAIF en esta capa
**Riesgo:** **exfiltración** de datos/modelos y acceso excesivo (demasiados permisos).
**Controles:** acceso mínimo por bucket/prefijo · versionado + *checksum* (integridad) · URLs prefirmadas con expiración.
**Elementos SAIF:** bases sólidas · coherencia de controles.

### Qué vas a hacer
1. Ver la **versión vulnerable** y por qué es un riesgo.
2. Aplicar el **control principal** (guiado, ya resuelto como ejemplo).
3. Completar los **`# TODO`** sobre tu propio modelo.
4. Escribir una **reflexión** (amenaza mitigada ↔ elemento de SAIF).

> Corre de punta a punta con **uv**. Es el único de los seis notebooks que necesita un servicio: **MinIO**.

### Dependencias y MinIO
```bash
uv add boto3 requests pandas scikit-learn xgboost joblib
```
Igual que en el Mini-TP 6, uso el **MinIO del stack de MLOps1_final** (contenedor `mlops-minio`,
API S3 en `localhost:9000`, credenciales `minioadmin`), en vez de levantar otro con `docker run`.
El bucket `datalake-seg-tp7` es propio de este Mini-TP, así no se mezcla con los buckets del TP
integrador.

Para la parte de **acceso mínimo** creo usuarios y políticas **de verdad** en ese MinIO, con el
cliente `mc` que ya trae el contenedor (`docker exec mlops-minio mc ...`). Todos se llaman
`tp7-*` y la última celda del notebook los borra.

In [1]:
import os, io, json, hashlib, secrets, subprocess, datetime, time
import requests, boto3, joblib
from botocore.client import Config
from botocore.exceptions import ClientError

ENDPOINT = "http://localhost:9000"
CONTENEDOR = "mlops-minio"
BUCKET = "datalake-seg-tp7"
# Credenciales de ADMINISTRADOR: solo para preparar el lake (crear bucket, usuarios y políticas).
# Ningún servicio las usa: cada uno tiene su propio usuario (TODO 3).
ROOT_USER = os.environ.get("MINIO_ROOT_USER", "minioadmin")
ROOT_PASS = os.environ.get("MINIO_ROOT_PASSWORD", "minioadmin123")

def cliente_s3(usuario, clave):
    return boto3.client("s3", endpoint_url=ENDPOINT, aws_access_key_id=usuario, aws_secret_access_key=clave,
                        region_name="us-east-1", config=Config(signature_version="s3v4"))

admin = cliente_s3(ROOT_USER, ROOT_PASS)
try:
    print("MinIO OK. Buckets existentes:", [b["Name"] for b in admin.list_buckets()["Buckets"]])
except Exception as e:
    raise RuntimeError("MinIO no responde en localhost:9000: levantá el stack de MLOps1_final") from e

MinIO OK. Buckets existentes: ['datalake', 'datalake-seg-tp7', 'datalake-tp6', 'federado', 'mlflow-artifacts', 'raw-data']


## 0. Mi modelo
Igual que en los Mini-TP 1 a 6 (y en los otros notebooks de este Mini-TP 7), uso **mi propio
modelo**: el XGBoost **`xgb_best`** del TP integrador, que predice el **precio de autos usados**
(dataset Car Dekho). Lo entreno acá con los mismos pasos e hiperparámetros del DAG de Airflow,
reusando `MLOps1_final/common` **sin copiarlo**, para no depender de MLflow.

In [2]:
import os, sys, time, json, hashlib, datetime
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

# Reusar el preprocesamiento del TP integrador (MLOps1_final/common)
TP_MLOPS1 = os.path.abspath(os.path.join("..", "TP_Final", "MLOps1_final"))
if TP_MLOPS1 not in sys.path:
    sys.path.insert(0, TP_MLOPS1)
DATASET_CSV = os.path.join(TP_MLOPS1, "dataset", "Car details v3.csv")

from common.constants import XGB_PARAMS, TRAIN_TEST_SIZE, TRAIN_TEST_SPLIT_SEED, MLFLOW_MODEL_NAME
from common.preprocessing import (
    extraer_columnas_tecnicas, drop_missing_technical_rows, drop_km_outliers,
    compute_top_brands, encode_categoricals, get_all_feature_columns,
    align_columns, build_feature_matrix, build_inference_features,
)
from common.schemas import CarRawInput

# Dataset y split estratificado por rango de precio (igual que en el TP integrador)
df = pd.read_csv(DATASET_CSV).drop_duplicates().reset_index(drop=True)
price_bins = pd.qcut(df["selling_price"], q=5,
                     labels=["muy_barato", "barato", "intermedio", "caro", "muy_caro"])
df_train_raw, df_test_raw = train_test_split(df, test_size=TRAIN_TEST_SIZE,
                                             stratify=price_bins, random_state=TRAIN_TEST_SPLIT_SEED)

def preparar(df_tr_raw, df_te_raw):
    """Feature engineering del TP integrador: devuelve X/y de train y test + metadata."""
    df_tr = drop_km_outliers(drop_missing_technical_rows(extraer_columnas_tecnicas(df_tr_raw)))
    df_te = drop_missing_technical_rows(extraer_columnas_tecnicas(df_te_raw))
    brands = compute_top_brands(df_tr)
    tr_enc, te_enc = encode_categoricals(df_tr, df_te, brands)
    feats = get_all_feature_columns(tr_enc)
    tr_enc, te_enc = align_columns(tr_enc, te_enc, feats)
    return (build_feature_matrix(tr_enc, feats), np.log1p(tr_enc["selling_price"]),
            build_feature_matrix(te_enc, feats), te_enc["selling_price"], brands, feats, df_tr, df_te)

X_train, y_train, X_test, y_test, top_brands, all_features, df_train, df_test = preparar(df_train_raw, df_test_raw)

modelo = XGBRegressor(**XGB_PARAMS)
modelo.fit(X_train, y_train)          # el modelo predice log(precio)

pred_test = np.expm1(modelo.predict(X_test))
METRICAS = {"r2": float(r2_score(y_test, pred_test)), "mae": float(mean_absolute_error(y_test, pred_test))}
MODEL_VERSION = f"{MLFLOW_MODEL_NAME}-local"
print(f"Modelo {MLFLOW_MODEL_NAME} listo. R2 test = {METRICAS['r2']:.4f} | MAE test = {METRICAS['mae']:,.0f}"
      f" | n_features = {len(all_features)}")

def predecir_precio(car: CarRawInput) -> float:
    """Mismo camino que la API del TP integrador: auto crudo -> features -> precio."""
    fila = build_inference_features(car, top_brands, all_features)
    return float(np.expm1(modelo.predict(fila)[0]))

COLS_AUTO = ["name", "year", "km_driven", "fuel", "seller_type", "transmission", "owner",
             "mileage", "engine", "max_power", "torque", "seats"]
# Auto de ejemplo (primer auto del dataset, precio real 450.000)
AUTO = {
    "name": "Maruti Swift Dzire VDI", "year": 2014, "km_driven": 145500,
    "fuel": "Diesel", "seller_type": "Individual", "transmission": "Manual",
    "owner": "First Owner", "mileage": "23.4 kmpl", "engine": "1248 CC",
    "max_power": "74 bhp", "torque": "190Nm@ 2000rpm", "seats": 5,
}
print(f"Precio predicho para el auto de ejemplo: {predecir_precio(CarRawInput(**AUTO)):,.0f}")

Modelo xgb_best listo. R2 test = 0.9372 | MAE test = 72,408 | n_features = 23
Precio predicho para el auto de ejemplo: 455,294


## 1. Versión VULNERABLE — bucket público, una sola credencial para todo, sin checksum
Así queda un lake "que funciona" pero sin seguridad:
- El bucket tiene una política de **lectura pública** ("para que el dashboard pueda bajar cosas").
- **Todos** los servicios usan la credencial de administrador.
- El modelo se sube y se carga sin verificar nada.

Uso un bucket aparte, `datalake-tp7-vulnerable`, que borro al final de esta sección.

In [3]:
BUCKET_V = "datalake-tp7-vulnerable"
if BUCKET_V not in [b["Name"] for b in admin.list_buckets()["Buckets"]]:
    admin.create_bucket(Bucket=BUCKET_V)
admin.put_bucket_policy(Bucket=BUCKET_V, Policy=json.dumps({
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"AWS": ["*"]}, "Action": ["s3:GetObject"],
                   "Resource": [f"arn:aws:s3:::{BUCKET_V}/*"]}]}))

buf = io.BytesIO(); joblib.dump(modelo, buf)
admin.put_object(Bucket=BUCKET_V, Key="models/v1/xgb_best.joblib", Body=buf.getvalue())
admin.put_object(Bucket=BUCKET_V, Key="raw/car_details_v3.csv", Body=df.to_csv(index=False).encode())

# a) Exfiltración: cualquiera, SIN credenciales, baja el modelo y el dataset
for key in ("models/v1/xgb_best.joblib", "raw/car_details_v3.csv"):
    r = requests.get(f"{ENDPOINT}/{BUCKET_V}/{key}")
    print(f"GET anónimo {key:28s} -> {r.status_code} ({len(r.content):,} bytes)")

robado = joblib.load(io.BytesIO(requests.get(f"{ENDPOINT}/{BUCKET_V}/models/v1/xgb_best.joblib").content))
print(f"Con el modelo robado, el atacante predice: {float(np.expm1(robado.predict(build_inference_features(CarRawInput(**AUTO), top_brands, all_features))[0])):,.0f}")

# b) Manipulación del artefacto: como todos usan la credencial de admin, cualquier servicio
#    comprometido puede reemplazar el modelo. Un .joblib/.pkl puede ejecutar código al cargarse.
class ArtefactoMalicioso:
    def __reduce__(self):            # pickle llama a esto al cargar: ejecuta lo que devuelve
        return (print, ("    !!! código del atacante ejecutado DENTRO del servicio al cargar el modelo !!!",))

buf = io.BytesIO(); joblib.dump(ArtefactoMalicioso(), buf)
admin.put_object(Bucket=BUCKET_V, Key="models/v1/xgb_best.joblib", Body=buf.getvalue())

print("\nEl servicio de predicción se reinicia y carga 'su' modelo desde el lake:")
cargado = joblib.load(io.BytesIO(admin.get_object(Bucket=BUCKET_V, Key="models/v1/xgb_best.joblib")["Body"].read()))
print(f"    modelo cargado: {cargado!r}  <- el original se perdió (sin versionado) y el código ya corrió")

# limpieza del bucket vulnerable (no dejar un bucket público dando vueltas)
for obj in admin.list_objects_v2(Bucket=BUCKET_V).get("Contents", []):
    admin.delete_object(Bucket=BUCKET_V, Key=obj["Key"])
admin.delete_bucket_policy(Bucket=BUCKET_V); admin.delete_bucket(Bucket=BUCKET_V)
print(f"\nBucket {BUCKET_V} borrado.")

GET anónimo models/v1/xgb_best.joblib    -> 200 (1,456,989 bytes)
GET anónimo raw/car_details_v3.csv       -> 200 (899,069 bytes)
Con el modelo robado, el atacante predice: 455,294

El servicio de predicción se reinicia y carga 'su' modelo desde el lake:
    !!! código del atacante ejecutado DENTRO del servicio al cargar el modelo !!!
    modelo cargado: None  <- el original se perdió (sin versionado) y el código ya corrió

Bucket datalake-tp7-vulnerable borrado.


## 2. Integridad — subir el modelo con checksum y verificarlo al cargar
Bucket nuevo, **privado** y con **versionado** activado. Subo **mi** modelo y su SHA-256, y la
carga verifica el checksum **antes** de deserializar: si el archivo fue cambiado, ni se abre.

In [4]:
if BUCKET not in [b["Name"] for b in admin.list_buckets()["Buckets"]]:
    admin.create_bucket(Bucket=BUCKET)
admin.put_bucket_versioning(Bucket=BUCKET, VersioningConfiguration={"Status": "Enabled"})
print("Versionado:", admin.get_bucket_versioning(Bucket=BUCKET)["Status"])

buf = io.BytesIO(); joblib.dump(modelo, buf); blob = buf.getvalue()
sha = hashlib.sha256(blob).hexdigest()
VERSIONES_BUENAS = {
    "models/v1/modelo.joblib": admin.put_object(Bucket=BUCKET, Key="models/v1/modelo.joblib", Body=blob)["VersionId"],
    "models/v1/modelo.sha256": admin.put_object(Bucket=BUCKET, Key="models/v1/modelo.sha256", Body=sha.encode())["VersionId"],
}

def cargar_verificando(s3=admin):
    blob = s3.get_object(Bucket=BUCKET, Key="models/v1/modelo.joblib")["Body"].read()
    esperado = s3.get_object(Bucket=BUCKET, Key="models/v1/modelo.sha256")["Body"].read().decode()
    if hashlib.sha256(blob).hexdigest() != esperado:
        raise ValueError("INTEGRIDAD ROTA: el artefacto fue manipulado")
    return joblib.load(io.BytesIO(blob))

m = cargar_verificando()
print("Modelo cargado y verificado por checksum. SHA:", sha[:20], "...")

# El mismo ataque de la sección 1, ahora contra el bucket con checksum
buf = io.BytesIO(); joblib.dump(ArtefactoMalicioso(), buf)
admin.put_object(Bucket=BUCKET, Key="models/v1/modelo.joblib", Body=buf.getvalue())
try:
    cargar_verificando()
except ValueError as e:
    print("Ataque 1 (cambia solo el modelo):", e, "-> no se ejecutó nada")

# Pero si el atacante puede escribir el modelo, también puede escribir el .sha256 de al lado
admin.put_object(Bucket=BUCKET, Key="models/v1/modelo.sha256", Body=hashlib.sha256(buf.getvalue()).hexdigest().encode())
print("Ataque 2 (cambia modelo Y checksum): ")
cargar_verificando()
print("    -> pasó la verificación: el checksum al lado del modelo no alcanza (ver TODO 1 y 3)")

# Con versionado, nada se pierde: restauro las versiones buenas (copiándolas como versión actual)
for key, version_buena in VERSIONES_BUENAS.items():
    admin.copy_object(Bucket=BUCKET, Key=key, CopySource={"Bucket": BUCKET, "Key": key, "VersionId": version_buena})
versiones = [v for v in admin.list_object_versions(Bucket=BUCKET, Prefix="models/v1/modelo.joblib")["Versions"]]
cargar_verificando()
print(f"Restaurado desde el versionado ({len(versiones)} versiones guardadas del modelo): original verificado OK")

Versionado: Enabled
Modelo cargado y verificado por checksum. SHA: b57b544436aaa0ecddef ...
Ataque 1 (cambia solo el modelo): INTEGRIDAD ROTA: el artefacto fue manipulado -> no se ejecutó nada
Ataque 2 (cambia modelo Y checksum): 
    !!! código del atacante ejecutado DENTRO del servicio al cargar el modelo !!!
    -> pasó la verificación: el checksum al lado del modelo no alcanza (ver TODO 1 y 3)
Restaurado desde el versionado (12 versiones guardadas del modelo): original verificado OK


## 3. Control principal (GUIADO) — URL prefirmada con expiración (en vez de bucket público)

In [5]:
# En vez de exponer el bucket, se entrega un enlace temporal y acotado:
url = admin.generate_presigned_url("get_object",
        Params={"Bucket": BUCKET, "Key": "models/v1/modelo.joblib"},
        ExpiresIn=60)   # válido 60 segundos
print("URL prefirmada (expira en 60 s):")
print(url[:90], "...")

# Probarla de verdad
r = requests.get(url)
print(f"\nGET con la URL prefirmada        -> {r.status_code}, sha ok: {hashlib.sha256(r.content).hexdigest() == sha}")
print(f"GET anónimo al mismo objeto      -> {requests.get(f'{ENDPOINT}/{BUCKET}/models/v1/modelo.joblib').status_code} (el bucket es privado)")
otra = url.replace("models/v1/modelo.joblib", "models/v1/modelo.sha256")
print(f"URL editada para otro objeto     -> {requests.get(otra).status_code} (la firma cubre bucket + key)")
corta = admin.generate_presigned_url("get_object", Params={"Bucket": BUCKET, "Key": "models/v1/modelo.joblib"}, ExpiresIn=1)
time.sleep(2)
print(f"URL vencida (expiraba en 1 s)    -> {requests.get(corta).status_code}")

# Política de acceso MÍNIMO (ejemplo conceptual): solo lectura de models/ para un rol.
politica = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow", "Action": ["s3:GetObject"],
        "Resource": f"arn:aws:s3:::{BUCKET}/models/*"   # nada más que models/
    }]
}
print("\nPolítica de acceso mínimo (solo lectura de models/):")
print(json.dumps(politica, indent=2))

URL prefirmada (expira en 60 s):
http://localhost:9000/datalake-seg-tp7/models/v1/modelo.joblib?X-Amz-Algorithm=AWS4-HMAC-S ...

GET con la URL prefirmada        -> 200, sha ok: True
GET anónimo al mismo objeto      -> 403 (el bucket es privado)
URL editada para otro objeto     -> 403 (la firma cubre bucket + key)
URL vencida (expiraba en 1 s)    -> 403

Política de acceso mínimo (solo lectura de models/):
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "s3:GetObject"
      ],
      "Resource": "arn:aws:s3:::datalake-seg-tp7/models/*"
    }
  ]
}


## 4. Tu entrega — completa los `# TODO`

### TODO 3 — zonas con permisos distintos (lo hago primero: los otros dos lo usan)
Cuatro usuarios de MinIO, uno por servicio de mi plataforma, cada uno con **solo** lo que necesita:

| Usuario | Servicio del TP | Puede | No puede |
|---|---|---|---|
| `tp7-ingestor` | streaming / carga de datos | **escribir** `raw/` | leer nada, tocar `models/` |
| `tp7-entrenador` | DAG de Airflow | leer `raw/` y `curated/`, escribir `curated/`, `models/` y `registro/` | borrar |
| `tp7-serving` | REST / GraphQL / gRPC | **leer** `models/` y `registro/` (también versiones viejas) | escribir nada, leer datos |
| `tp7-analista` | dashboards | leer `curated/` | leer `models/` (no puede llevarse el modelo) |

Las políticas son JSON de IAM (el mismo formato que AWS S3) y MinIO las aplica de verdad.
Nadie tiene `s3:DeleteObject`, y con versionado, aunque alguien sobrescriba un objeto, la versión
anterior queda.

### TODO 1 — mi modelo con checksum, pero el checksum en un lugar aparte
El problema de la sección 2 es que el `.sha256` vive **al lado** del modelo, con los mismos
permisos. Lo resuelvo así:
- El entrenador sube el modelo a `models/v1/` y registra en **`registro/xgb_best/v1.json`** el
  SHA-256, el tamaño, la **`VersionId`** de S3 del objeto y la metadata para predecir.
- El serving lee el registro, baja **esa versión exacta** (por `VersionId`), verifica el SHA-256
  **antes** de deserializar, y recién ahí carga y predice.
- Si alguien sobrescribe `models/v1/xgb_best.joblib`, se crea una versión nueva, pero el serving
  sigue bajando la registrada. Para cambiar lo que se sirve hay que escribir en `registro/`, y eso
  solo lo puede hacer el entrenador.

Es la misma idea que el Model Registry de MLflow del TP integrador: el registro dice **qué
versión exacta** está en producción.

### TODO 2 — registro de acceso (gobernanza)
Cada servicio usa el lake a través de un `LakeAuditado`, que registra cada `get`/`put` con fecha
y hora, usuario, operación, key, versión y resultado (incluidos los `AccessDenied`). En
producción esto se complementa con el *audit log* del propio MinIO (`mc admin trace` o un
webhook de auditoría), que registra también lo que no pasa por mi código.

In [6]:
# ---------- TODO 3: usuarios y políticas reales en MinIO ----------
def mc(*args, entrada=None):
    r = subprocess.run(["docker", "exec", "-i", CONTENEDOR, "mc", *args], input=entrada,
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"mc {' '.join(args[:3])}: {r.stderr.strip() or r.stdout.strip()}")
    return r.stdout.strip()

def arn(*prefijos):
    return [f"arn:aws:s3:::{BUCKET}/{p}*" for p in prefijos]

POLITICAS = {
    "tp7-ingestor":   [{"Effect": "Allow", "Action": ["s3:PutObject"], "Resource": arn("raw/")}],
    "tp7-entrenador": [{"Effect": "Allow", "Action": ["s3:GetObject"], "Resource": arn("raw/", "curated/")},
                       {"Effect": "Allow", "Action": ["s3:PutObject"], "Resource": arn("curated/", "models/", "registro/")}],
    "tp7-serving":    [{"Effect": "Allow", "Action": ["s3:GetObject", "s3:GetObjectVersion"],
                        "Resource": arn("models/", "registro/")}],
    "tp7-analista":   [{"Effect": "Allow", "Action": ["s3:GetObject"], "Resource": arn("curated/")}],
}

mc("alias", "set", "tp7", "http://localhost:9000", ROOT_USER, ROOT_PASS)
CLIENTES = {}
for usuario, statements in POLITICAS.items():
    clave = secrets.token_urlsafe(24)           # secreto nuevo en cada corrida, nunca en el código
    subprocess.run(["docker", "exec", "-i", CONTENEDOR, "sh", "-c", f"cat > /tmp/{usuario}.json"],
                   input=json.dumps({"Version": "2012-10-17", "Statement": statements}), text=True, check=True)
    mc("admin", "policy", "create", "tp7", usuario, f"/tmp/{usuario}.json")
    mc("admin", "user", "add", "tp7", usuario, clave)
    try:
        mc("admin", "policy", "attach", "tp7", usuario, "--user", usuario)
    except RuntimeError as e:
        if "already" not in str(e).lower():
            raise
    CLIENTES[usuario] = cliente_s3(usuario, clave)
    print(f"Usuario {usuario:15s} creado con su política: "
          + "; ".join(f"{'/'.join(a.split(':')[-1] for a in s['Action'])} en "
                      f"{', '.join(r.split('/', 1)[1] for r in s['Resource'])}" for s in statements))
time.sleep(1)    # MinIO propaga las políticas

# ---------- TODO 2: registro de acceso ----------
ACCESOS = []

class LakeAuditado:
    """Envoltorio del cliente S3 que registra cada operación (y su resultado)."""
    def __init__(self, usuario):
        self.usuario, self.s3 = usuario, CLIENTES[usuario]

    def _registrar(self, op, key, version=None, resultado="OK"):
        ACCESOS.append({"ts": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
                        "usuario": self.usuario, "op": op, "key": key,
                        "version": (version or "")[:8] or None, "resultado": resultado})

    def put(self, key, body):
        try:
            r = self.s3.put_object(Bucket=BUCKET, Key=key, Body=body)
            self._registrar("put", key, r.get("VersionId")); return r.get("VersionId")
        except ClientError as e:
            self._registrar("put", key, resultado=e.response["Error"]["Code"]); raise

    def get(self, key, version=None):
        try:
            kw = {"VersionId": version} if version else {}
            r = self.s3.get_object(Bucket=BUCKET, Key=key, **kw)
            self._registrar("get", key, r.get("VersionId")); return r["Body"].read()
        except ClientError as e:
            self._registrar("get", key, version, resultado=e.response["Error"]["Code"]); raise

    def delete(self, key):
        try:
            self.s3.delete_object(Bucket=BUCKET, Key=key); self._registrar("delete", key)
        except ClientError as e:
            self._registrar("delete", key, resultado=e.response["Error"]["Code"]); raise

ingestor, entrenador = LakeAuditado("tp7-ingestor"), LakeAuditado("tp7-entrenador")
serving, analista = LakeAuditado("tp7-serving"), LakeAuditado("tp7-analista")

Usuario tp7-ingestor    creado con su política: PutObject en raw/*
Usuario tp7-entrenador  creado con su política: GetObject en raw/*, curated/*; PutObject en curated/*, models/*, registro/*
Usuario tp7-serving     creado con su política: GetObject/GetObjectVersion en models/*, registro/*
Usuario tp7-analista    creado con su política: GetObject en curated/*


In [7]:
# ---------- TODO 1: flujo completo con mi modelo ----------
# 1) El ingestor sube el dataset crudo a raw/
ingestor.put("raw/car_dekho/car_details_v3.csv", df.to_csv(index=False).encode())

# 2) El entrenador lee raw/, publica curated/ y sube el modelo + su entrada en el registro
crudo = pd.read_csv(io.BytesIO(entrenador.get("raw/car_dekho/car_details_v3.csv")))
curado = pd.concat([X_train.assign(selling_price=np.expm1(y_train).round().values, split="train"),
                    X_test.assign(selling_price=y_test.values, split="test")], ignore_index=True)
buf = io.BytesIO(); curado.to_parquet(buf, index=False)
entrenador.put("curated/car_dekho/car_features.parquet", buf.getvalue())

buf = io.BytesIO(); joblib.dump(modelo, buf); blob = buf.getvalue()
version_id = entrenador.put("models/v1/xgb_best.joblib", blob)
registro = {"modelo": "xgb_best", "version": "v1", "key": "models/v1/xgb_best.joblib",
            "version_id": version_id, "sha256": hashlib.sha256(blob).hexdigest(), "bytes": len(blob),
            "top_brands": top_brands, "all_features": all_features, "metricas": METRICAS,
            "creado": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}
entrenador.put("registro/xgb_best/v1.json", json.dumps(registro, indent=2).encode())
print(f"Modelo publicado: {registro['key']} (VersionId {version_id[:8]}..., sha {registro['sha256'][:16]}...)")

# 3) El serving carga desde el registro: versión exacta + checksum ANTES de deserializar
def cargar_desde_registro(lake, version="v1"):
    reg = json.loads(lake.get(f"registro/xgb_best/{version}.json"))
    blob = lake.get(reg["key"], version=reg["version_id"])
    if hashlib.sha256(blob).hexdigest() != reg["sha256"]:
        raise ValueError("INTEGRIDAD ROTA: el artefacto no coincide con el registro")
    return joblib.load(io.BytesIO(blob)), reg

def predecir_con(m, reg, auto):
    fila = build_inference_features(CarRawInput(**auto), reg["top_brands"], reg["all_features"])
    return float(np.expm1(m.predict(fila)[0]))

m_serv, reg = cargar_desde_registro(serving)
print(f"Serving cargó la versión registrada y verificada -> precio del auto de ejemplo: {predecir_con(m_serv, reg, AUTO):,.0f}")

Modelo publicado: models/v1/xgb_best.joblib (VersionId 603efb26..., sha b57b544436aaa0ec...)
Serving cargó la versión registrada y verificada -> precio del auto de ejemplo: 455,294


### Pruebas de permisos y de integridad

In [8]:
def intento(lake, op, key, *args):
    try:
        getattr(lake, op)(key, *args); return "OK"
    except ClientError as e:
        return e.response["Error"]["Code"]

casos = [
    # (usuario, operación, key, esperado, por qué importa)
    (ingestor,   "put",    "raw/car_dekho/nuevos.csv",       "OK",           "su trabajo"),
    (ingestor,   "get",    "raw/car_dekho/car_details_v3.csv", "AccessDenied", "no lee nada"),
    (ingestor,   "put",    "models/v1/xgb_best.joblib",      "AccessDenied", "no puede tocar el modelo"),
    (entrenador, "get",    "raw/car_dekho/car_details_v3.csv", "OK",         "su trabajo"),
    (entrenador, "delete", "models/v1/xgb_best.joblib",      "AccessDenied", "nadie borra"),
    (serving,    "get",    "models/v1/xgb_best.joblib",      "OK",           "su trabajo"),
    (serving,    "put",    "models/v1/xgb_best.joblib",      "AccessDenied", "comprometido, no puede alterar el modelo"),
    (serving,    "put",    "registro/xgb_best/v1.json",      "AccessDenied", "ni el registro"),
    (serving,    "get",    "raw/car_dekho/car_details_v3.csv", "AccessDenied", "no puede exfiltrar datos"),
    (analista,   "get",    "curated/car_dekho/car_features.parquet", "OK",   "su trabajo"),
    (analista,   "get",    "models/v1/xgb_best.joblib",      "AccessDenied", "no puede llevarse el modelo"),
    (analista,   "get",    "raw/car_dekho/car_details_v3.csv", "AccessDenied", "no ve datos crudos"),
]
ok = True
for lake, op, key, esperado, motivo in casos:
    args = (b"x",) if op == "put" else ()
    res = intento(lake, op, key, *args); ok &= res == esperado
    print(f"  {lake.usuario:15s} {op:6s} {key:40s} {res:13s} {'OK' if res == esperado else 'FALLA'}  ({motivo})")

# Una URL prefirmada nunca da más permisos que los de quien la firma
url_analista = CLIENTES["tp7-analista"].generate_presigned_url(
    "get_object", Params={"Bucket": BUCKET, "Key": "models/v1/xgb_best.joblib"}, ExpiresIn=60)
url_serving = CLIENTES["tp7-serving"].generate_presigned_url(
    "get_object", Params={"Bucket": BUCKET, "Key": "models/v1/xgb_best.joblib"}, ExpiresIn=60)
ra, rs = requests.get(url_analista).status_code, requests.get(url_serving).status_code
print(f"\n  URL prefirmada por el analista para el modelo -> {ra} | firmada por el serving -> {rs}")
ok &= (ra, rs) == (403, 200)

# Ataque: un entrenador comprometido sobrescribe el modelo con el artefacto malicioso
buf = io.BytesIO(); joblib.dump(ArtefactoMalicioso(), buf)
version_maliciosa = entrenador.put("models/v1/xgb_best.joblib", buf.getvalue())
m2, reg2 = cargar_desde_registro(serving)
print(f"\n  Modelo sobrescrito: el serving sigue cargando la VersionId registrada -> precio {predecir_con(m2, reg2, AUTO):,.0f}")

# ... y si además cambia la VersionId en el registro, el checksum no coincide
reg_falso = {**registro, "version_id": version_maliciosa}
entrenador.put("registro/xgb_best/v1.json", json.dumps(reg_falso).encode())
try:
    cargar_desde_registro(serving); ok = False
except ValueError as e:
    print(f"  Registro apuntando a la versión maliciosa -> {e} (no se ejecutó nada)")
# (si el atacante también recalcula el sha del registro, solo queda el versionado + la auditoría:
#  por eso el registro debería estar firmado o, mejor, en MLflow con permisos propios)
entrenador.put("registro/xgb_best/v1.json", json.dumps(registro, indent=2).encode())   # restauro
print("\nTodas las pruebas OK" if ok else "\nHAY PRUEBAS QUE FALLARON")

  tp7-ingestor    put    raw/car_dekho/nuevos.csv                 OK            OK  (su trabajo)
  tp7-ingestor    get    raw/car_dekho/car_details_v3.csv         AccessDenied  OK  (no lee nada)
  tp7-ingestor    put    models/v1/xgb_best.joblib                AccessDenied  OK  (no puede tocar el modelo)
  tp7-entrenador  get    raw/car_dekho/car_details_v3.csv         OK            OK  (su trabajo)
  tp7-entrenador  delete models/v1/xgb_best.joblib                AccessDenied  OK  (nadie borra)
  tp7-serving     get    models/v1/xgb_best.joblib                OK            OK  (su trabajo)
  tp7-serving     put    models/v1/xgb_best.joblib                AccessDenied  OK  (comprometido, no puede alterar el modelo)
  tp7-serving     put    registro/xgb_best/v1.json                AccessDenied  OK  (ni el registro)
  tp7-serving     get    raw/car_dekho/car_details_v3.csv         AccessDenied  OK  (no puede exfiltrar datos)
  tp7-analista    get    curated/car_dekho/car_features.parquet

In [9]:
# TODO 2: registro de acceso (en producción: append-only, fuera del alcance de los propios servicios)
pd.set_option("display.width", 200)
acc = pd.DataFrame(ACCESOS)
print("Accesos por usuario y resultado:")
print(acc.pivot_table(index="usuario", columns="resultado", values="op", aggfunc="count", fill_value=0))
acc[acc["resultado"] != "OK"]

Accesos por usuario y resultado:
resultado       AccessDenied  OK
usuario                         
tp7-analista               2   1
tp7-entrenador             1   8
tp7-ingestor               2   2
tp7-serving                3   7


,ts,usuario,op,key,version,resultado
8,2026-10-10T19:08:17+00:00,tp7-ingestor,get,raw/car_dekho/car_details_v3.csv,None,AccessDenied
9,2026-10-10T19:08:17+00:00,tp7-ingestor,put,models/v1/xgb_best.joblib,None,AccessDenied
11,2026-10-10T19:08:17+00:00,tp7-entrenador,delete,models/v1/xgb_best.joblib,None,AccessDenied
13,2026-10-10T19:08:17+00:00,tp7-serving,put,models/v1/xgb_best.joblib,None,AccessDenied
14,2026-10-10T19:08:17+00:00,tp7-serving,put,registro/xgb_best/v1.json,None,AccessDenied
15,2026-10-10T19:08:17+00:00,tp7-serving,get,raw/car_dekho/car_details_v3.csv,None,AccessDenied
17,2026-10-10T19:08:17+00:00,tp7-analista,get,models/v1/xgb_best.joblib,None,AccessDenied
18,2026-10-10T19:08:17+00:00,tp7-analista,get,raw/car_dekho/car_details_v3.csv,None,AccessDenied


In [10]:
# Limpieza: borro los usuarios y políticas tp7-* del MinIO compartido.
# El bucket datalake-seg-tp7 queda (como datalake-tp6 en el Mini-TP 6), privado y versionado.
for usuario in POLITICAS:
    for cmd in (["admin", "user", "rm", "tp7", usuario], ["admin", "policy", "rm", "tp7", usuario]):
        try:
            mc(*cmd)
        except RuntimeError as e:
            print("aviso:", e)
quedan = mc("admin", "user", "list", "tp7")
mc("alias", "rm", "tp7")
subprocess.run(["docker", "exec", CONTENEDOR, "sh", "-c", "rm -f /tmp/tp7-*.json"], check=False)
print("Usuarios y políticas tp7-* borrados. Usuarios que quedan en MinIO:", quedan or "(ninguno)")

Usuarios y políticas tp7-* borrados. Usuarios que quedan en MinIO: (ninguno)


## 5. Reflexión

**1. ¿Por qué una URL prefirmada con expiración es más segura que un bucket público?**
Porque da acceso a **un solo objeto**, **por un tiempo corto**, y **solo a quien recibe el
enlace**. Con el bucket público de la sección 1, cualquiera bajaba el modelo y el dataset
completos, sin credenciales y para siempre. Con la URL prefirmada probé que:
- no sirve para otro objeto (si se edita la key, la firma no coincide: 403);
- deja de servir cuando vence (403);
- nunca da más permisos que los de quien la firma: la que firmó el analista para el modelo da
  403, porque el analista no puede leer `models/`.
El bucket sigue privado, y cada URL que se entrega queda asociada a un usuario y a un momento.

**2. ¿Qué ataque detecta el checksum del modelo?**
La **manipulación del artefacto**: que alguien reemplace el modelo por otro (uno sesgado, uno con
*backdoor*, o directamente un archivo que ejecuta código al cargarse, como mostré en la sección 1:
un `.joblib` es un *pickle*, y cargarlo **ejecuta** lo que el atacante quiera). El checksum se
verifica **antes** de deserializar, así que el archivo alterado ni se abre.
Pero vi que el checksum **solo sirve si el atacante no puede cambiarlo también**: con el
`.sha256` al lado del modelo y los mismos permisos, el ataque 2 pasó. Por eso lo puse en
`registro/` (que solo escribe el entrenador), junto con la `VersionId` exacta. Y el versionado
hace que nada se pierda: el original se restauró.

**3. ¿Qué elemento de SAIF cubre mi control principal?**

| Control | Amenaza | Elemento de SAIF |
|---|---|---|
| Bucket privado + URLs prefirmadas con expiración | exfiltración de datos y del modelo | **Bases sólidas de seguridad** |
| Un usuario por servicio con acceso mínimo por prefijo | acceso excesivo, servicio comprometido | **Bases sólidas de seguridad** |
| Checksum + `VersionId` en un registro aparte | manipulación del artefacto (incluido código malicioso) | **Bases sólidas** (cadena de suministro del modelo) |
| Versionado del bucket, sin permisos de borrado | pérdida o sobrescritura del modelo | **Detectar y actuar** (poder volver atrás) |
| Mismas reglas para los 4 servicios del TP (REST, GraphQL, gRPC, streaming leen el mismo lake) | una capa con más permisos que las otras | **Coherencia de controles** |
| Registro de accesos (incluidos los `AccessDenied`) | uso indebido sin rastro | **Detectar y actuar** + gobernanza |

El control principal (acceso mínimo + integridad + URLs prefirmadas) cubre **bases sólidas**:
son controles clásicos de almacenamiento aplicados a lo más valioso del sistema, los datos y el
modelo. Y cubre **coherencia de controles** porque el lake es el punto que comparten todas las
capas: en el TP integrador, REST, GraphQL, gRPC y streaming leen el mismo modelo. Si uno solo de
esos servicios tuviera la credencial de administrador (como en la sección 1), comprometerlo
alcanzaría para cambiar el modelo que sirven **todos**.